# 1. Install and initialize

### Cell 1 — Dependencies

In [ ]:
!pip -q install pandas numpy requests \
matplotlib seaborn scipy \
scikit-learn biopython

### Cell 2 — Project initialization

In [ ]:
from pathlib import Path
import os
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET

BASE = Path("/content/ASTRABIO_03")

FOLDERS = [
    "data",
    "literature",
    "countermeasures",
    "results",
    "reports",
    "figures"
]

for folder in FOLDERS:
    (BASE / folder).mkdir(
        parents=True,
        exist_ok=True
    )

print("ASTRABIO Notebook 03 initialized")

# 2. Connect Notebook 02

### Cell 3 — Import previous model metadata

In [ ]:
NOTEBOOK_02 = Path("/content/ASTRABIO_02")

MODEL_METADATA = (
    NOTEBOOK_02 /
    "models" /
    "model_metadata.json"
)

if MODEL_METADATA.exists():

    with open(MODEL_METADATA) as f:
        model_info = json.load(f)

    print("Notebook 02 connected")
    print(json.dumps(model_info, indent=4))

else:

    model_info = {}

    print("Notebook 02 metadata not found")
    print("Upload or connect Notebook 02 results.")

### Cell 4 — Import biological pathway results

In [ ]:
PATHWAY_FOLDER = (
    NOTEBOOK_02 /
    "results" /
    "pathways"
)

pathway_files = []

if PATHWAY_FOLDER.exists():

    pathway_files = list(
        PATHWAY_FOLDER.glob("*.csv")
    )

if pathway_files:

    pathway_df = pd.read_csv(
        pathway_files[0]
    )

    display(pathway_df.head(20))

else:

    pathway_df = pd.DataFrame()

    print("No pathway results found.")

# 3. Identify biological research targets

### Cell 5 — Configure research targets

In [ ]:
TARGET_GENES = []

TARGET_PATHWAYS = []

BIOLOGICAL_CONDITIONS = [
    "spaceflight",
    "microgravity",
    "radiation",
    "oxidative stress",
    "immune dysregulation",
    "muscle atrophy",
    "bone loss",
    "mitochondrial dysfunction"
]

print("Research conditions:")
for condition in BIOLOGICAL_CONDITIONS:
    print("-", condition)

### Cell 6 — Extract genes from pathway results

In [ ]:
def extract_genes_from_pathways(df):

    genes = set()

    if df.empty:
        return []

    possible_columns = [
        "Genes",
        "genes",
        "gene_symbols",
        "Overlap"
    ]

    for column in possible_columns:

        if column in df.columns:

            for value in df[column].dropna():

                for gene in str(value).split(";"):

                    gene = gene.strip()

                    if gene:
                        genes.add(gene)

            break

    return sorted(genes)


if not pathway_df.empty:

    discovered_genes = (
        extract_genes_from_pathways(
            pathway_df
        )
    )

    TARGET_GENES = discovered_genes

    print("Discovered genes:", len(TARGET_GENES))
    print(TARGET_GENES[:50])

# 4. Scientific literature discovery

### Cell 7 — Configure PubMed API

In [ ]:
from Bio import Entrez

Entrez.email = "your_email@example.com"

PUBMED_BASE = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
)

Replace the email with your own email address for NCBI identification.

### Cell 8 — Search scientific literature

In [ ]:
def search_pubmed(
    query,
    max_results=50
):

    url = PUBMED_BASE + "esearch.fcgi"

    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json",
        "sort": "relevance"
    }

    response = requests.get(
        url,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    return data["esearchresult"]["idlist"]


SEARCH_TERMS = [
    "spaceflight human physiology countermeasures",
    "microgravity human muscle bone countermeasures",
    "space radiation biological countermeasures",
    "spaceflight immune system human research",
    "microgravity oxidative stress mitochondria"
]

all_pmids = set()

for term in SEARCH_TERMS:

    try:

        ids = search_pubmed(
            term,
            max_results=50
        )

        all_pmids.update(ids)

        print(
            "Retrieved:",
            len(ids),
            "for:",
            term
        )

    except Exception as e:

        print("Search error:", term, e)

print("Total unique publications:", len(all_pmids))

### Cell 9 — Download publication abstracts

In [ ]:
def fetch_pubmed_articles(pmids):

    if not pmids:
        return pd.DataFrame()

    url = PUBMED_BASE + "efetch.fcgi"

    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "rettype": "abstract",
        "retmode": "xml"
    }

    response = requests.get(
        url,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    root = ET.fromstring(response.content)

    articles = []

    for article in root.findall(".//PubmedArticle"):

        pmid = article.findtext(
            ".//PMID",
            default=""
        )

        title = article.findtext(
            ".//ArticleTitle",
            default=""
        )

        abstract_parts = []

        for item in article.findall(
            ".//Abstract/AbstractText"
        ):

            abstract_parts.append(
                "".join(item.itertext())
            )

        abstract = " ".join(abstract_parts)

        year = article.findtext(
            ".//PubDate/Year",
            default=""
        )

        articles.append({
            "PMID": pmid,
            "Title": title,
            "Year": year,
            "Abstract": abstract,
            "URL": (
                "https://pubmed.ncbi.nlm.nih.gov/"
                f"{pmid}/"
            )
        })

    return pd.DataFrame(articles)


literature_df = fetch_pubmed_articles(
    list(all_pmids)
)

display(literature_df.head(10))

literature_df.to_csv(
    BASE / "literature" / "pubmed_articles.csv",
    index=False
)

# 5. Countermeasure research database

### Cell 10 — Define research categories

In [ ]:
COUNTERMEASURE_CATEGORIES = {

    "Muscle and bone": [
        "muscle atrophy",
        "bone loss",
        "resistance exercise",
        "mechanical loading"
    ],

    "Radiation biology": [
        "radiation damage",
        "DNA damage",
        "oxidative stress",
        "radioprotection"
    ],

    "Immune system": [
        "immune dysregulation",
        "inflammation",
        "immune function"
    ],

    "Cardiovascular": [
        "cardiovascular deconditioning",
        "fluid redistribution",
        "vascular function"
    ],

    "Metabolic and mitochondrial": [
        "mitochondrial dysfunction",
        "metabolic adaptation",
        "oxidative stress"
    ]
}

for category, targets in COUNTERMEASURE_CATEGORIES.items():

    print("\n", category)

    for target in targets:
        print(" -", target)

### Cell 11 — Create candidate research database

In [ ]:
countermeasures = pd.DataFrame([

    {
        "candidate": "Resistance exercise",
        "category": "Muscle and bone",
        "target": "Muscle atrophy",
        "mechanism": "Mechanical loading",
        "research_status": "Research candidate"
    },

    {
        "candidate": "Aerobic exercise",
        "category": "Cardiovascular",
        "target": "Cardiovascular deconditioning",
        "mechanism": "Cardiovascular conditioning",
        "research_status": "Research candidate"
    },

    {
        "candidate": "Nutritional strategies",
        "category": "Muscle and bone",
        "target": "Bone and muscle changes",
        "mechanism": "Nutritional support",
        "research_status": "Research candidate"
    },

    {
        "candidate": "Radiation shielding",
        "category": "Radiation biology",
        "target": "Radiation exposure",
        "mechanism": "Physical exposure reduction",
        "research_status": "Engineering countermeasure"
    },

    {
        "candidate": "Sleep and circadian interventions",
        "category": "Immune system",
        "target": "Circadian disruption",
        "mechanism": "Circadian regulation",
        "research_status": "Research candidate"
    },

    {
        "candidate": "Mitochondrial protection research",
        "category": "Metabolic and mitochondrial",
        "target": "Mitochondrial dysfunction",
        "mechanism": "Mitochondrial homeostasis",
        "research_status": "Experimental research"
    }

])

display(countermeasures)

# 6. Literature and countermeasure matching

### Cell 12 — Match publications to biological conditions

In [ ]:
def match_literature(
    literature,
    conditions
):

    results = []

    for _, article in literature.iterrows():

        text = (
            str(article["Title"]) + " " +
            str(article["Abstract"])
        ).lower()

        matches = []

        for condition in conditions:

            if condition.lower() in text:
                matches.append(condition)

        results.append({

            "PMID": article["PMID"],
            "Title": article["Title"],
            "Year": article["Year"],
            "Matched_conditions": ", ".join(matches),
            "Match_count": len(matches),
            "URL": article["URL"]

        })

    return pd.DataFrame(results)


matched_literature = match_literature(
    literature_df,
    BIOLOGICAL_CONDITIONS
)

matched_literature = matched_literature.sort_values(
    "Match_count",
    ascending=False
)

display(matched_literature.head(20))

### Cell 13 — Match countermeasure candidates

In [ ]:
def match_countermeasures(
    articles,
    candidates
):

    results = []

    for _, candidate in candidates.iterrows():

        target = candidate["target"].lower()

        mechanism = candidate["mechanism"].lower()

        matches = []

        for _, article in articles.iterrows():

            text = (
                str(article["Title"]) + " " +
                str(article["Abstract"])
            ).lower()

            if (
                target in text
                or mechanism in text
            ):

                matches.append(
                    article["PMID"]
                )

        results.append({

            "candidate": candidate["candidate"],
            "category": candidate["category"],
            "target": candidate["target"],
            "matching_publications": len(matches),
            "PMIDs": ";".join(matches)

        })

    return pd.DataFrame(results)


countermeasure_matches = match_countermeasures(
    literature_df,
    countermeasures
)

display(countermeasure_matches)

This is keyword-based screening, not proof that a candidate works. Each matched publication needs manual scientific review.

# 7. Scientific evidence assessment

### Cell 14 — Classify publication types

In [ ]:
def classify_study_type(text):

    text = str(text).lower()

    if any(x in text for x in [
        "randomized controlled trial",
        "randomised controlled trial"
    ]):
        return "Randomized controlled trial"

    if any(x in text for x in [
        "systematic review",
        "meta-analysis",
        "meta analysis"
    ]):
        return "Systematic review / Meta-analysis"

    if any(x in text for x in [
        "clinical trial",
        "clinical study"
    ]):
        return "Clinical research"

    if any(x in text for x in [
        "mouse model",
        "mice",
        "rat model",
        "murine"
    ]):
        return "Animal research"

    if any(x in text for x in [
        "cell culture",
        "in vitro",
        "cell line"
    ]):
        return "In vitro research"

    if any(x in text for x in [
        "astronaut",
        "spaceflight crew",
        "space flight crew"
    ]):
        return "Human spaceflight research"

    return "Other / Unclassified"


literature_df["Study_Type"] = (
    literature_df["Title"].fillna("") + " " +
    literature_df["Abstract"].fillna("")
).apply(classify_study_type)

display(
    literature_df[
        ["PMID", "Title", "Year", "Study_Type"]
    ].head(20)
)

### Cell 15 — Evidence summary

In [ ]:
evidence_summary = (
    literature_df
    .groupby("Study_Type")
    .size()
    .reset_index(name="Publication_Count")
    .sort_values(
        "Publication_Count",
        ascending=False
    )
)

display(evidence_summary)

plt.figure(figsize=(11, 6))

sns.barplot(
    data=evidence_summary,
    x="Publication_Count",
    y="Study_Type",
    hue="Study_Type",
    legend=False
)

plt.title(
    "ASTRABIO Scientific Literature Evidence Categories"
)

plt.xlabel("Number of Publications")
plt.ylabel("Study Type")

plt.tight_layout()
plt.show()

### Cell 16 — Research limitations and evidence review

In [ ]:
def create_evidence_review(articles):

    review = articles.copy()

    review["Review_Status"] = "Requires manual review"

    review["Human_Spaceflight_Relevant"] = (
        review["Study_Type"].eq(
            "Human spaceflight research"
        )
    )

    review["Has_Abstract"] = (
        review["Abstract"]
        .fillna("")
        .str.len() > 100
    )

    review["Evidence_Limitation"] = (
        "Publication category and abstract "
        "do not establish intervention efficacy."
    )

    return review


evidence_review = create_evidence_review(
    literature_df
)

display(evidence_review.head(20))

# 8. Biological mechanism mapping

### Cell 17 — Create biological mechanism map

In [ ]:
mechanism_map = pd.DataFrame([

    {
        "Biological_condition": "Muscle atrophy",
        "Potential_mechanism": "Altered mechanical loading",
        "Research_direction": "Exercise countermeasures"
    },

    {
        "Biological_condition": "Bone loss",
        "Potential_mechanism": "Altered bone remodeling",
        "Research_direction": "Mechanical loading and nutrition"
    },

    {
        "Biological_condition": "Oxidative stress",
        "Potential_mechanism": "Reactive oxygen species imbalance",
        "Research_direction": "Redox biology"
    },

    {
        "Biological_condition": "Immune dysregulation",
        "Potential_mechanism": "Altered immune signaling",
        "Research_direction": "Immune regulation research"
    },

    {
        "Biological_condition": "Mitochondrial dysfunction",
        "Potential_mechanism": "Altered mitochondrial homeostasis",
        "Research_direction": "Mitochondrial biology"
    },

    {
        "Biological_condition": "Radiation damage",
        "Potential_mechanism": "DNA damage and cellular stress",
        "Research_direction": "Radiation protection research"
    }

])

display(mechanism_map)

mechanism_map.to_csv(
    BASE / "results" / "biological_mechanisms.csv",
    index=False
)

# 9. Research prioritization engine

### Cell 18 — Calculate research relevance scores

This prioritizes topics for further investigation based on literature relevance, not treatment effectiveness.

In [ ]:
def prioritize_research(
    matches,
    articles
):

    study_counts = (
        articles.groupby("PMID")["Study_Type"]
        .first()
        .to_dict()
    )

    results = []

    for _, row in matches.iterrows():

        pmids = [
            p for p in str(row["PMIDs"]).split(";")
            if p
        ]

        human_count = sum(
            study_counts.get(p) ==
            "Human spaceflight research"
            for p in pmids
        )

        systematic_count = sum(
            study_counts.get(p) ==
            "Systematic review / Meta-analysis"
            for p in pmids
        )

        results.append({

            "candidate": row["candidate"],

            "publication_count": len(pmids),

            "human_spaceflight_publications": human_count,

            "systematic_review_publications": systematic_count,

            "research_priority_score": (
                len(pmids)
                + 3 * human_count
                + 2 * systematic_count
            )

        })

    result = pd.DataFrame(results)

    return result.sort_values(
        "research_priority_score",
        ascending=False
    )


research_priorities = prioritize_research(
    countermeasure_matches,
    literature_df
)

display(research_priorities)

### Cell 19 — Visualize research priorities

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=research_priorities,
    x="research_priority_score",
    y="candidate",
    hue="candidate",
    legend=False
)

plt.title(
    "ASTRABIO Countermeasure Research Priorities"
)

plt.xlabel("Research Relevance Score")
plt.ylabel("Candidate")

plt.tight_layout()

plt.savefig(
    BASE / "figures" / "research_priorities.png",
    dpi=300
)

plt.show()

# 10. ASTRABIO research assistant

### Cell 20 — Interactive research query

In [ ]:
def astrabio_research_assistant(question):

    question = question.lower()

    results = []

    for _, article in literature_df.iterrows():

        text = (
            str(article["Title"]) + " " +
            str(article["Abstract"])
        ).lower()

        query_terms = [
            word for word in re.findall(
                r"\b[a-z0-9]+\b",
                question
            )
            if len(word) > 3
        ]

        score = sum(
            term in text
            for term in query_terms
        )

        if score > 0:

            results.append({
                "PMID": article["PMID"],
                "Title": article["Title"],
                "Year": article["Year"],
                "Relevance": score,
                "URL": article["URL"]
            })

    results_df = pd.DataFrame(results)

    if results_df.empty:

        print("No matching publications found.")

        return results_df

    results_df = results_df.sort_values(
        "Relevance",
        ascending=False
    )

    display(results_df.head(15))

    return results_df


import re

# Example:
research_results = astrabio_research_assistant(
    "microgravity muscle atrophy astronaut exercise"
)

# 11. Generate research reports

### Cell 21 — Export research database

In [ ]:
countermeasures.to_csv(
    BASE / "countermeasures" /
    "countermeasure_database.csv",
    index=False
)

literature_df.to_csv(
    BASE / "literature" /
    "literature_database.csv",
    index=False
)

research_priorities.to_csv(
    BASE / "results" /
    "research_priorities.csv",
    index=False
)

evidence_review.to_csv(
    BASE / "results" /
    "evidence_review.csv",
    index=False
)

print("Research databases exported")

### Cell 22 — Generate Markdown research report

In [ ]:
report_path = (
    BASE / "reports" /
    "ASTRABIO_03_RESEARCH_REPORT.md"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "# ASTRABIO Countermeasure Research Report\n\n"
    )

    f.write(
        "## Research overview\n\n"
    )

    f.write(
        f"- Publications retrieved: {len(literature_df)}\n"
    )

    f.write(
        f"- Research candidates: {len(countermeasures)}\n"
    )

    f.write(
        f"- Biological conditions: {len(BIOLOGICAL_CONDITIONS)}\n"
    )

    f.write(
        "\n## Research priorities\n\n"
    )

    f.write(
        research_priorities.to_markdown(index=False)
    )

    f.write(
        "\n\n## Evidence categories\n\n"
    )

    f.write(
        evidence_summary.to_markdown(index=False)
    )

    f.write(
        "\n\n## Scientific limitations\n\n"
    )

    f.write(
        "These outputs are research hypotheses and "
        "literature-screening results. They do not "
        "establish clinical efficacy, safety, causality "
        "or astronaut treatment recommendations.\n"
    )

print("Report generated:", report_path)

### Cell 23 — Export complete ASTRABIO 03 project

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/content/ASTRABIO_03_RESULTS",
    "zip",
    BASE
)

from google.colab import files

files.download(archive_path)